# ĐỒ ÁN CUỐI KHÓA: XÂY DỰNG, TỐI ƯU VÀ CĂN CHỈNH MÔ HÌNH NGÔN NGỮ LỚN TỪ SỐ 0
## Capstone Project: Building, Optimizing, and Aligning Large Language Models from Scratch

### Phần 1: Khối Kiến Trúc Lõi Bắt Buộc (Mandatory Core Foundation)
1. **Tự xây dựng Tokenizer:** Byte-level BPE Tokenizer from scratch (0% OOV, hỗ trợ UTF-8, tiếng Việt, code, emojis).
2. **Kiến trúc Transformer Decoder-Only Hiện đại:** Pre-RMSNorm, Rotary Position Embedding (RoPE), SwiGLU, và **DeepSeek Multi-Head Latent Attention (MLA)**.
3. **Hệ thống Quản lý Bộ nhớ & Dynamic KV Cache:** Cấu trúc dữ liệu lưu trữ Dynamic Cache với 2 pha Prefill ($O(L)$) và Decode ($O(1)$ per token).
4. **Profiling Thực nghiệm:** Đo đạc TTFT, TPOT, Throughput (tokens/s) và VRAM Footprint khi Bật vs. Tắt Cache trên các mốc $L \in \{128, 512, 1024, 2048\}$.

## 1. Khởi Tạo Môi Trường & Thiết Lập Hạt Giống Tái Lập (Reproducibility)

In [ ]:
import os
import random
import sys
import time
from typing import List, Tuple, Dict, Optional

# Thêm đường dẫn project vào sys.path
sys.path.append(os.path.abspath("."))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Đặt seed cố định để tái lập kết quả thực nghiệm chuẩn xác
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Execution Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Tiêu Chí 1: Xây Dựng Byte-level BPE Tokenizer Từ Số 0
- Bắt đầu từ 256 bytes cơ bản của UTF-8.
- Tách từ bằng regex phân tách ranh giới hình vị.
- Đếm tần suất cặp byte liền kề và thực hiện merge lặp lại.
- Hỗ trợ Special Tokens: `<|pad|>`, `<|bos|>`, `<|eos|>`, `<|think|>`, `</|think|>`. Tỷ lệ OOV tuyệt đối bằng 0%.

In [ ]:
from core.tokenizer import ByteLevelBPETokenizer, SpecialToken

# Tập ngữ liệu huấn luyện mẫu (Anh - Việt - Logic CoT - Emojis)
sample_corpus = [
    "Xin chào! Đây là mô hình ngôn ngữ lớn được xây dựng từ số 0 hoàn toàn.",
    "Kiến trúc Transformer Decoder-Only kết hợp Multi-Head Latent Attention (MLA).",
    "DeepSeek-V3 và DeepSeek-R1 tối ưu hóa KV Cache bằng cơ chế nén Low-Rank.",
    "Trí tuệ nhân tạo đang phát triển vượt bậc với khả năng suy luận từng bước.",
    "<|think|> Phân tích câu hỏi: Để tối ưu VRAM, ta cần nén Key và Value trước khi lưu cache. </|think|> Kết luận chính xác! 🚀",
    "Toán học: f(x) = x^2 + 2x + 1 = (x + 1)^2. Kết quả cuối cùng là 42.",
]

# Khởi tạo và huấn luyện Tokenizer
tokenizer = ByteLevelBPETokenizer()
target_vocab = 400
tokenizer.train(sample_corpus, vocab_size=target_vocab, min_frequency=1)

print(f"✅ Kích thước từ điển đã huấn luyện: {tokenizer.vocab_size} tokens")

# Kiểm thử Encode & Decode với chuỗi ký tự đa dạng
test_sentence = "Xin chào DeepSeek MLA! 🚀 <|think|> Tư duy suy luận </|think|>"
encoded_ids = tokenizer.encode(test_sentence)
decoded_sentence = tokenizer.decode(encoded_ids)

print(f"\nChuỗi gốc:   {test_sentence}")
print(f"Token IDs:   {encoded_ids}")
print(f"Giải mã lại: {decoded_sentence}")
assert test_sentence == decoded_sentence, "Lỗi giải mã BPE!"
print("✅ Kiểm thử BPE Tokenizer hoàn tất: Giải mã trùng khớp 100% không mất mát!")

## 3. Tiêu Chí 2: Khối Kiến Trúc Lõi Hiện Đại (RMSNorm, RoPE, SwiGLU)
- **Pre-RMSNorm:** Chuẩn hóa theo căn bậc hai bình phương trung bình, bỏ qua mean-centering giúp tăng tốc 15%.
- **RoPE (Rotary Position Embedding):** Xoay vector Query và Key trong không gian 2D, không chiếm tham số học, tự nhiên bảo toàn vị trí tương đối.
- **SwiGLU FFN:** Cổng kích hoạt phi tuyến tính $(\text{SiLU}(x W_{\text{gate}}) \odot x W_{\text{up}}) W_{\text{down}}$.

In [ ]:
from core.layers import RMSNorm, RotaryEmbedding, SwiGLUFFN

batch, seq, d_model = 2, 8, 256
dummy_x = torch.randn(batch, seq, d_model, device=device)

# 1. Test RMSNorm
rmsnorm = RMSNorm(d_model, device=device)
normed_x = rmsnorm(dummy_x)
print(f"RMSNorm Output Shape: {normed_x.shape}")

# 2. Test RoPE
rope = RotaryEmbedding(dim=64, max_seq_len=512, device=device)
pos_ids = torch.arange(seq, device=device).unsqueeze(0).expand(batch, -1)
cos, sin = rope(dummy_x, pos_ids)
print(f"RoPE Cos/Sin Shape: {cos.shape}")

# 3. Test SwiGLU
swiglu = SwiGLUFFN(hidden_dim=d_model, intermediate_dim=512, device=device)
ffn_out = swiglu(normed_x)
print(f"SwiGLU Output Shape: {ffn_out.shape}")
print("✅ Toàn bộ các khối nền tảng Pre-RMSNorm, RoPE, SwiGLU hoạt động hoàn hảo!")

## 4. Multi-Head Latent Attention (MLA) & Dynamic KV Cache
DeepSeek MLA nén toàn bộ không gian Key-Value xuống vector tiềm ẩn $c^{KV}$ và tách biệt RoPE qua $k^R$.
Dynamic KV Cache chỉ cần lưu $[c^{KV}, k^R]$ $\rightarrow$ Tiết kiệm tới **86% VRAM** so với MHA truyền thống!

In [ ]:
from configs.model_config import LLMConfig
from core.model import ModernDecoderOnlyLLM

# Cấu hình mô hình chuẩn phục vụ nghiệm thu cốt lõi tại máy local
config = LLMConfig(
    vocab_size=1000,
    hidden_dim=256,
    num_layers=4,
    num_heads=4,
    intermediate_dim=512,
    max_seq_len=2048,
    attention_type="mla",
    kv_lora_rank=64,       # d_c: Nén Key-Value latent
    qk_rope_head_dim=32,   # d_r: Decoupled RoPE Key
    head_dim=64,
)

model = ModernDecoderOnlyLLM(config, device=device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Mô hình ModernDecoderOnlyLLM (MLA) khởi tạo thành công!")
print(f"Tổng số tham số: {total_params:,} tham số")

# Forward pass thử nghiệm tính Cross-Entropy Loss
input_ids = torch.randint(0, config.vocab_size, (2, 16), device=device)
logits, loss = model(input_ids, labels=input_ids)
print(f"Logits Shape: {logits.shape}")
print(f"Cross-Entropy Loss: {loss.item():.4f}")

## 5. Sinh Tự Hồi Quy (Autoregressive Generation) & Kiểm Thử Tính Nhất Quán Của Cache
So sánh quá trình sinh từ khi **Bật Cache** (Prefill 1 lần + Decode từng token) vs. **Tắt Cache** (Feed lại toàn bộ lịch sử mỗi bước).
Xác nhận tính đúng đắn toán học: Cả 2 cách phải cho ra **chính xác 100% cùng 1 chuỗi token** khi Greedy Sampling.

In [ ]:
prompt = torch.randint(0, config.vocab_size, (1, 8), device=device)

# 1. Sinh từ với KV Cache BẬT
gen_cached = model.generate(
    prompt, max_new_tokens=12, temperature=0.0, use_cache=True
)

# 2. Sinh từ với KV Cache TẮT
gen_uncached = model.generate(
    prompt, max_new_tokens=12, temperature=0.0, use_cache=False
)

print(f"Chuỗi sinh (Cache BẬT): {gen_cached.tolist()[0]}")
print(f"Chuỗi sinh (Cache TẮT): {gen_uncached.tolist()[0]}")

assert torch.equal(gen_cached, gen_uncached), "Lỗi: Cache và Non-cache cho kết quả khác nhau!"
print("✅ Nghiệm thu hoàn tất: Kết quả sinh từ BẬT Cache hoàn toàn trùng khớp với TẮT Cache!")

## 6. Tiêu Chí 3: Đo Đạc & Lập Biểu Đồ Thực Nghiệm (Profiling Suite)
Đo đạc 4 chỉ số thực nghiệm bắt buộc trên các độ dài ngữ cảnh tăng dần:
$$L \in \{128, 512, 1024, 2048\}$$
1. **First Token Latency (TTFT)**
2. **Next Token Latency (TPOT)**
3. **Tốc độ sinh (Tokens/giây)**
4. **Mức độ chiếm dụng VRAM**

In [ ]:
from profiling.benchmark_cache import run_kv_cache_benchmark

# Chạy thực nghiệm profiling tự động
profile_results = run_kv_cache_benchmark(
    config=config,
    context_lengths=(128, 512, 1024, 2048),
    num_gen_tokens=16,
    output_image_path="profiling/kv_cache_benchmark.png",
    device=device,
)

# Hiển thị biểu đồ kết quả trực tiếp trong Notebook
from IPython.display import Image, display
display(Image(filename="profiling/kv_cache_benchmark.png"))